# Case study: preserved sp-SVC analysis workflow for Visium HD data

## Notebook Guide

**Purpose.** Preserve and explain the Visium HD sp-SVC analysis workflow while keeping its approved bounded-audit status explicit.

**Inputs.** Use either the Application inputs required for reconstruction or the released VisiumHD reconstructed H5AD described below.

### Two ways to obtain the analysis inputs

1. **Reconstruct locally.** Download the [Application input data](https://zenodo.org/records/17705737), place `P1CRC_HD.h5ad` and the matched reference under `raw_data/Real_application/`, and run `python reconstruct.py --config configs/application/VisiumHD.yaml` from the repository root.
2. **Start from released results.** Download the [REVISE-reconstructed sp-SVC VisiumHD data](https://zenodo.org/records/18389835) and place the reconstructed object at `output/sp_SVC_case/P1CRC/sp_SVC.h5ad`. Then continue with the analysis workflow below.

**Reconstruction input.** The route starts from `raw_data/Real_application/P1CRC_HD.h5ad` and the matched single-cell reference. The spatial object provides counts and `obsm["spatial"]`; the reference provides shared genes and broad `Level1` labels.

**Reconstruction command.** `python reconstruct.py --config configs/application/VisiumHD.yaml`

**Expected reconstructed outputs and analysis input.** `output/sp_SVC_case/P1CRC/sp_SVC.h5ad`. Its status in this checkout remains `not_available`; historical tables are not substituted for the missing H5AD.

**Analysis workflow.** Inspect the raw input, compare clustering metrics and groupwise spatial autocorrelation, score an EMT gene set, and visualize marker programs when the declared reconstructed input exists.

**Outputs.** Metric, spatial-autocorrelation, clustering, pathway, and marker artifacts are written under `../../output/sp_SVC_analysis/` by the preserved workflow.

**Execution and evidence boundary.** This notebook has not been fully executed in the current closure. Evidence is limited to static old-to-new mapping, deterministic Scanpy/Squidpy fixture parity, historical-output continuity, and a bounded raw-input AUCell smoke. The downstream scientific narrative is preserved as historical workflow/paper context and does not establish fresh compute, reconstruction parity, or biological validation.


## Minimum REVISE input schema for this case

To run this Visium HD-style `sp-SVC` case, users only need two `AnnData` files. The spatial file should store a bin-or-pseudo-cell-by-gene count matrix in `st_adata.X`, gene names in `st_adata.var_names`, and two-dimensional spatial coordinates in `st_adata.obsm["spatial"]`. The scRNA-seq reference should store a cell-by-gene count matrix in `sc_ref_adata.X`, gene names in `sc_ref_adata.var_names`, and broad cell-type labels in `sc_ref_adata.obs["Level1"]`. The two files only need to share genes by name; route-specific preprocessing handles normalization and gene overlap.

`Level2` labels and image-derived metadata can improve case-specific downstream interpretation, but they are not part of the minimum input needed to start a REVISE application run. Data acquisition and the exact local paths are defined once in the Notebook Guide above.


## First glimpse at raw data

## Spatial Autocorrelation

Compare Moran-style spatial autocorrelation summaries between raw and sp-SVC outputs.


In [ ]:
import os
os.environ.setdefault("TQDM_DISABLE", "1")
os.environ.setdefault("TQDM_MININTERVAL", "60")

try:
    from IPython import get_ipython
    _ipython = get_ipython()
    if _ipython is not None:
        _ipython.run_line_magic("matplotlib", "inline")
except Exception:
    pass

import os

import scanpy as sc
import pandas as pd
import numpy as np

from tqdm import tqdm as _tqdm

def tqdm(iterable=None, *args, **kwargs):
    kwargs.setdefault("disable", True)
    return _tqdm(iterable, *args, **kwargs)
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

from revise.analysis.basic.unsupervised import compute_leiden_sweep
from revise.analysis.basic.spatial_autocorrelation import compute_groupwise_spatial_autocorrelation
from revise.analysis.basic.gene_set_scoring import read_gmt
from revise.analysis.advanced.aucell import score_gene_set_aucell

def sample_by_identity(adata, sample_size, seed):
    sample_size = min(sample_size, adata.n_obs)
    sampled_names = np.random.RandomState(seed).choice(adata.obs_names, size=sample_size, replace=False)
    return adata[sampled_names].copy()

def sample_paired_by_identity(original, reconstructed, sample_size, seed):
    if set(original.obs_names) != set(reconstructed.obs_names):
        raise ValueError("Original and reconstructed objects must have the same observation identities")
    sampled_names = np.random.RandomState(seed).choice(
        original.obs_names, size=min(sample_size, original.n_obs), replace=False
    )
    return original[sampled_names].copy(), reconstructed[sampled_names].copy()

def run_cluster_analysis(adata, save_path, resolutions, cell_type_col):
    os.makedirs(save_path, exist_ok=True)
    clustered, metrics = compute_leiden_sweep(
        adata, resolutions=resolutions, reference_col=cell_type_col, random_state=0
    )
    metrics.to_csv(f"{save_path}/metric.csv", index=False)
    colors = [f"leiden_res_{resolution}" for resolution in resolutions] + [cell_type_col]
    sc.tl.tsne(clustered, n_pcs=30, random_state=0)
    sc.pl.tsne(clustered, color=colors, ncols=2, show=False)
    plt.savefig(f"{save_path}/tsne.png")
    plt.close()
    sc.tl.umap(clustered)
    sc.pl.umap(clustered, color=colors, ncols=2, show=False)
    plt.savefig(f"{save_path}/umap.png")
    plt.close()
    for color_key in [f"leiden_res_{resolutions[1]}", cell_type_col]:
        sc.pl.scatter(clustered, size=30, color=color_key, x="x", y="y", show=False)
    plt.savefig(f"{save_path}/spatial.png")
    plt.close()

def save_groupwise_autocorrelation(adata, save_path, cell_type_col, mode="moran"):
    table = compute_groupwise_spatial_autocorrelation(
        adata, groupby=cell_type_col, mode=mode
    )
    table = table.fillna(0).sort_values("All", ascending=False)
    table.to_csv(f"{save_path}/{mode}_autocorr.csv")
    selected_genes = []
    for column in table.columns:
        selected_genes.extend(table[column].nlargest(10).index)
    selected = table.loc[list(dict.fromkeys(selected_genes))]
    sns.clustermap(selected, annot=False, cmap="coolwarm", row_cluster=False)
    plt.savefig(f"{save_path}/{mode}_heatmap.pdf", dpi=300)
    plt.close()
    return table

def plot_compare_spatial_autocorr_pdf(df_sp_svc, df_original, save_path, mode):
    cell_types = df_sp_svc.columns.intersection(df_original.columns)
    rows = []
    for cell_type in cell_types:
        rows.extend({'Cell Type': cell_type, 'Value': value, 'Source': 'SP_SVC'} for value in df_sp_svc[cell_type].dropna())
        rows.extend({'Cell Type': cell_type, 'Value': value, 'Source': 'Original'} for value in df_original[cell_type].dropna())
    plot_df = pd.DataFrame(rows)
    plot_df = plot_df[plot_df['Value'] < 0.8]
    plt.figure(figsize=(12, 6))
    sns.boxplot(x='Cell Type', y='Value', hue='Source', data=plot_df, palette=['#1f77b4', '#ff7f0e'])
    for index in range(len(cell_types) - 1):
        plt.axvline(x=index + 0.5, color='gray', linestyle='--', alpha=0.5)
    for index, cell_type in enumerate(cell_types):
        svc_values = plot_df[(plot_df['Cell Type'] == cell_type) & (plot_df['Source'] == 'SP_SVC')]['Value']
        original_values = plot_df[(plot_df['Cell Type'] == cell_type) & (plot_df['Source'] == 'Original')]['Value']
        if len(svc_values) > 0 and len(original_values) > 0:
            _, p_value = ttest_ind(svc_values, original_values, nan_policy='omit')
            if p_value < 0.05:
                stars = '*' if p_value >= 0.01 else '**' if p_value >= 0.001 else '***'
                text_y = max(svc_values.max(), original_values.max()) + 0.05 * (plot_df['Value'].max() - plot_df['Value'].min())
                plt.text(index, text_y, stars, ha='center', va='bottom', fontsize=12)
    plt.xticks(rotation=30, ha='right')
    plt.title(f'Spatial Autocorrelation ({mode}) Comparison by Cell Type')
    plt.tight_layout()
    plt.savefig(f"{save_path}/Compare_{mode}.pdf")
    plt.close()


## Analysis prerequisite checkpoint

Use one of the two acquisition routes in the Notebook Guide. Before running the code below, confirm that `../../raw_data/Real_application/P1CRC_HD.h5ad`, the matched reference, and `../../output/sp_SVC_case/P1CRC/sp_SVC.h5ad` resolve from `reproduce/case/`. The reconstructed H5AD is not present or validated in the current checkout, so the remaining cells are a preserved reference workflow rather than a newly executed result.


In [ ]:
patient_id = "P1CRC"
data_type = "HD"

raw_data_path = "../../raw_data/Real_application"

raw_file_name = f"{raw_data_path}/{patient_id}_{data_type}.h5ad"
sc_file_name = f"{raw_data_path}/adata_sc_all_reanno.h5ad"

In [ ]:
adata = sc.read(raw_file_name)
adata.obs['Level1'].value_counts()

## Clustering metric

In [ ]:
resolutions = [0.3, 0.5, 0.8]
patient_ids = ["P1CRC"] # one tumor sample for test
patient_ids = ["P1CRC", "P2CRC", "P5CRC"]
patient_ids = ["P1CRC"] # available HD input in this reproducibility environment


data_type = "HD"
cell_type_col = "Level1"

sample_size = 30000

raw_data_path = "../../raw_data/Real_application"
svc_data_path = "../../output/sp_SVC_case"

save_dir = "results/sp_SVC_case"
save_dir = "../../output/sp_SVC_analysis"


## Input And Cache Checks

Confirm that the original relative data paths resolve before running analysis cells.


In [ ]:
expected_result_suffixes = [
    "original/metric.csv",
    "original/moran_autocorr.csv",
    "sp_SVC/metric.csv",
    "sp_SVC/moran_autocorr.csv",
]

for patient_id in tqdm(patient_ids, desc="Processing patients"):
            
    save_path = f"{save_dir}/{patient_id}_{data_type}"
    os.makedirs(save_path, exist_ok=True)
    expected_result_files = [f"{save_path}/{suffix}" for suffix in expected_result_suffixes]
    if all(os.path.exists(path) for path in expected_result_files):
        print(f"Using cached core result files for {patient_id}_{data_type}")
        continue

    print(f"Processing original data for {patient_id}_{data_type}")
    adata_sp = sc.read(f"{raw_data_path}/{patient_id}_{data_type}.h5ad")
    adata_sp = adata_sp[adata_sp.obs[cell_type_col] != "Unknown"].copy()
    print(f"Processing SVC data for {patient_id}_{data_type}")
    adata_sp_svc = sc.read(f"{svc_data_path}/{patient_id}/sp_SVC.h5ad")
    adata_sp_svc = adata_sp_svc[adata_sp_svc.obs[cell_type_col]!= "Unknown"].copy()

    if sample_size is not None:
        adata_sp, adata_sp_svc = sample_paired_by_identity(
            adata_sp, adata_sp_svc, sample_size=sample_size, seed=0
        )

    run_cluster_analysis(adata_sp, f"{save_path}/original", resolutions, cell_type_col)
    all_sp = save_groupwise_autocorrelation(
        adata_sp, f"{save_path}/original", cell_type_col
    )
    run_cluster_analysis(adata_sp_svc, f"{save_path}/sp_SVC", resolutions, cell_type_col)
    all_sp_svc = save_groupwise_autocorrelation(
        adata_sp_svc, f"{save_path}/sp_SVC", cell_type_col
    )
    plot_compare_spatial_autocorr_pdf(all_sp_svc, all_sp, save_path, mode="moran")
    

## Plot summary ARI and NMI

## Metric Comparison

Compare clustering and reconstruction metrics between original and sp-SVC data.


In [ ]:
import pandas as pd

from tqdm import tqdm as _tqdm

def tqdm(iterable=None, *args, **kwargs):
    kwargs.setdefault("disable", True)
    return _tqdm(iterable, *args, **kwargs)

patient_ids = ["P1CRC"]
metrics = ["ARI", "NMI"]
data_type = "HD"
cell_type_col = "Level1"


In [ ]:
all_metric_dfs = []

for patient_id in tqdm(patient_ids):
    original_metric_df = pd.read_csv(f"../../output/sp_SVC_analysis/{patient_id}_{data_type}/original/metric.csv")
    select_res = original_metric_df.loc[original_metric_df['ARI'].idxmax(), "resolution"]
    print(patient_id, select_res)
    
    original_metric_df = original_metric_df[original_metric_df['resolution'] == select_res]
    original_metric_df['data_class'] = "Original"

    sp_SVC_metric_df = pd.read_csv(f"../../output/sp_SVC_analysis/{patient_id}_{data_type}/sp_SVC/metric.csv")
    sp_SVC_metric_df = sp_SVC_metric_df[sp_SVC_metric_df['resolution'] == select_res]
    sp_SVC_metric_df['data_class'] = "sp_SVC"
    
    metric_df = pd.concat([original_metric_df, sp_SVC_metric_df], axis=0)
    metric_df['patient_id'] = patient_id
    
    all_metric_dfs.append(metric_df)

final_metric_df = pd.concat(all_metric_dfs, axis=0)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Comparison of Original vs sp_SVC Metrics', fontsize=16, fontweight='bold')

axes = axes.flatten()

for i, patient_id in enumerate(patient_ids):
    patient_data = final_metric_df[final_metric_df['patient_id'] == patient_id]
    
    plot_data = []
    for metric in metrics:
        for data_class in ["Original", "sp_SVC"]:
            value = patient_data[patient_data['data_class'] == data_class][metric].values[0]
            plot_data.append({
                'Metric': metric,
                'Value': value,
                'Data Class': data_class
            })
    
    plot_df = pd.DataFrame(plot_data)
    
    ax = axes[i]
    bars = sns.barplot(data=plot_df, x='Metric', y='Value', hue='Data Class', ax=ax, legend=False)
    ax.set_title(f'Patient: {patient_id}')
    ax.set_ylim(0, 1)  
    ax.grid(True, alpha=0.3)
    
    for container in bars.containers:
        bars.bar_label(container, fmt='%.3f', padding=3)

# Add the legend in the last position (the sixth subplot)
axes[5].set_visible(False)  # Hide the axes of the sixth subplot

plt.tight_layout()
plt.subplots_adjust(top=0.93)  # Leave space for the overall title

plt.savefig(f'{save_dir}/metrics_comparison.pdf', dpi=300, bbox_inches='tight')
plt.show()


## Plot spatial correlation

### Heatmap of the moran'I metric in mean or quantile

In [ ]:
import pandas as pd
import numpy as np

import seaborn as sns
from matplotlib import pyplot as plt
from scipy.stats import ttest_ind
from tqdm import tqdm as _tqdm

def tqdm(iterable=None, *args, **kwargs):
    kwargs.setdefault("disable", True)
    return _tqdm(iterable, *args, **kwargs)

def plot_compare_spatial_autocorr_heatmap(df_sp_svc, df_original, save_dir, mode="moran", 
                                         statistic="mean", quantile=0.5, cmap="viridis"):
    """
    Generate a heatmap comparing two dataframes across cell types with significance testing.
    
    Parameters:
    df_sp_svc (pd.DataFrame): Dataframe with genes as rows, cell types as columns.
    df_original (pd.DataFrame): Dataframe with genes as rows, cell types as columns.
    save_dir (str): Directory to save the heatmap image.
    mode (str): Mode identifier for the output filename.
    statistic (str): Statistic to calculate - "mean" or "quantile".
    quantile (float): Quantile to calculate if statistic is "quantile" (0-1).
    cmap (str): Colormap for the heatmap.
    """
    
    # Prepare data for heatmap
    cell_types = df_sp_svc.columns.intersection(df_original.columns)
    
    # Calculate statistics and p-values
    stats_data = []
    p_values = []
    
    for cell_type in cell_types:
        # Extract values for each cell type
        sp_svc_values = df_sp_svc[cell_type].dropna()
        original_values = df_original[cell_type].dropna()
        
        # Calculate selected statistic
        if statistic == "mean":
            sp_svc_stat = sp_svc_values.mean()
            original_stat = original_values.mean()
        elif statistic == "quantile":
            sp_svc_stat = sp_svc_values.quantile(quantile)
            original_stat = original_values.quantile(quantile)
        else:
            raise ValueError("statistic must be 'mean' or 'quantile'")
        
        # Calculate p-value
        if len(sp_svc_values) > 1 and len(original_values) > 1:
            _, p_val = ttest_ind(sp_svc_values, original_values, nan_policy='omit')
        else:
            p_val = 1.0  # Not enough data for test
        
        stats_data.append([original_stat, sp_svc_stat])
        p_values.append(p_val)
    
    # Create dataframe for heatmap
    heatmap_df = pd.DataFrame(stats_data, 
                             index=cell_types, 
                             columns=['Original', 'SP_SVC'])
    
    # Create annotation matrix with significance stars
    annotation_matrix = []
    for p_val in p_values:
        if p_val < 0.05:
            sig = '*' if p_val >= 0.01 else '**' if p_val >= 0.001 else '***'
            annotation_matrix.append(['', sig])
        else:
            annotation_matrix.append(['', ''])
    
    # Create the heatmap
    plt.figure(figsize=(8, max(6, len(cell_types) * 0.5)))
    
    # Plot heatmap with specified colormap
    sns.heatmap(heatmap_df, 
                annot=np.array(annotation_matrix),  # Use annotations for significance
                fmt='',  # Empty format since we're using custom annotations
                cmap=cmap, 
                cbar_kws={'label': f'{statistic.capitalize()} Value'},
                linewidths=0.5,
                linecolor='gray',
                # vmin=-0.01,
                # vmax=0.1,
                annot_kws={'fontsize': 14, 'fontweight': 'bold'})
    
    plt.ylabel('')
    plt.xlabel('')
    plt.xticks([])
    
    # Adjust layout and save
    plt.tight_layout()
    plt.savefig(f"{save_dir}/Compare_{mode}_{statistic}_heatmap.pdf", dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()


In [ ]:
data_type = "HD"
mode = "moran"
patient_ids = ["P1CRC"]

save_dir = f"../../output/sp_SVC_analysis/{patient_id}_{data_type}"
for patient_id in tqdm(patient_ids, desc = "patient_id"):

    moranI_sp_file = f"{save_dir}/original/{mode}_autocorr.csv"
    moranI_sp = pd.read_csv(moranI_sp_file, index_col = 0)
    moranI_sp.columns = moranI_sp.columns.str.replace("/", "_")
    # moranI_sp
    moranI_sp_svc_file = f"{save_dir}/sp_SVC/{mode}_autocorr.csv"
    moranI_sp_svc = pd.read_csv(moranI_sp_svc_file, index_col = 0)
    # moranI_sp_svc

    plot_compare_spatial_autocorr_heatmap(moranI_sp_svc, moranI_sp, save_dir, mode="moran", 
                                         statistic="quantile", quantile=0.75, cmap="RdBu_r")

### Boxplot of the moran'I metric

In [ ]:
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from tqdm import tqdm as _tqdm

def tqdm(iterable=None, *args, **kwargs):
    kwargs.setdefault("disable", True)
    return _tqdm(iterable, *args, **kwargs)
from scipy.stats import ttest_ind

def plot_compare_spatial_autocorr(df_sp_svc, df_original, save_dir, mode = "moran"):
    """
    Generate a boxplot comparing two dataframes across cell types with significance testing.
    
    Parameters:
    df_sp_svc (pd.DataFrame): Dataframe with genes as rows, cell types as columns.
    df_original (pd.DataFrame): Dataframe with genes as rows, cell types as columns.
    output_file (str): Path to save the boxplot image.
    """
   
    # Prepare data for boxplot
    cell_types = df_sp_svc.columns.intersection(df_original.columns)
    data = []
    for cell_type in cell_types:
        # Extract values for each cell type
        sp_svc_values = df_sp_svc[cell_type].dropna()
        original_values = df_original[cell_type].dropna()
        # Add to data list with labels
        for val in sp_svc_values:
            data.append({'Cell Type': cell_type, 'Value': val, 'Source': 'SP_SVC'})
        for val in original_values:
            data.append({'Cell Type': cell_type, 'Value': val, 'Source': 'Original'})
    
    # Convert to dataframe
    plot_df = pd.DataFrame(data)
    
    # Set up the boxplot
    plt.figure(figsize=(12, 6))
    plot_df = plot_df[plot_df["Value"] < 0.8]
    sns.boxplot(x='Cell Type', y='Value', 
                hue='Source', data=plot_df, 
                palette=['#1f77b4', '#ff7f0e'], 
                legend=False)


    # Add dashed lines between cell types
    for i in range(len(cell_types) - 1):
        plt.axvline(x=i + 0.5, color='gray', linestyle='--', alpha=0.5)
    
    # Rotate x-axis labels
    plt.xticks(rotation=30, ha='right')
    
    # Add significance annotations
    for i, cell_type in enumerate(cell_types):
        sp_svc_vals = plot_df[(plot_df['Cell Type'] == cell_type) & (plot_df['Source'] == 'SP_SVC')]['Value']
        orig_vals = plot_df[(plot_df['Cell Type'] == cell_type) & (plot_df['Source'] == 'Original')]['Value']
        if len(sp_svc_vals) > 0 and len(orig_vals) > 0:
            t_stat, p_val = ttest_ind(sp_svc_vals, orig_vals, nan_policy='omit')
            # Add stars for significance
            if p_val < 0.05:
                sig = '*' if p_val >= 0.01 else '**' if p_val >= 0.001 else '***'
                max_y = max(sp_svc_vals.max(), orig_vals.max())
                text_y = max_y + 0.05 * (plot_df['Value'].max() - plot_df['Value'].min())
                text_y = 0.85
                plt.text(i, text_y, 
                        sig, ha='center', va='bottom', fontsize=20)
    
    # Adjust layout and save
    plt.xlabel('')
    plt.ylabel('')
    plt.xticks([])
    plt.yticks(fontsize = 20)
    
    # Get the current Axes object
    ax = plt.gca()
    # Set the outer border line width
    for spine in ax.spines.values():
        spine.set_linewidth(2)

    ax.tick_params(axis='y', which='major', width=2, length = 8)

    plt.tight_layout()
    plt.savefig(f"{save_dir}/Compare_{mode}.png")
    plt.show()


In [ ]:
data_type = "HD"
mode = "moran"
patient_ids = ["P1CRC"]

save_dir = f"../../output/sp_SVC_analysis/{patient_id}_{data_type}"

for patient_id in tqdm(patient_ids, desc = "patient_id"):

    moranI_sp_file = f"{save_dir}/original/{mode}_autocorr.csv"
    moranI_sp = pd.read_csv(moranI_sp_file, index_col = 0)
    moranI_sp.columns = moranI_sp.columns.str.replace("/", "_")
    # moranI_sp
    moranI_sp_svc_file = f"{save_dir}/sp_SVC/{mode}_autocorr.csv"
    moranI_sp_svc = pd.read_csv(moranI_sp_svc_file, index_col = 0)
    # moranI_sp_svc

    plot_compare_spatial_autocorr(moranI_sp_svc, moranI_sp, save_dir = save_dir, mode = "moran")

## Analysis CAF surrounded tumor in P1CRC

sp-SVC recovers spatially organized EMT-associated transcriptional programs and identifies genes linked to clinical significance

In [ ]:
import scanpy as sc

raw_data_path = "../../raw_data/Real_application"
svc_data_path = "../../output/sp_SVC_case"

patient_id = "P1CRC"
data_type = "HD"

raw_file_name = f"{raw_data_path}/{patient_id}_{data_type}.h5ad"
sp_SVC_file_name = f"{svc_data_path}/{patient_id}/sp_SVC.h5ad"
adata_sp = sc.read_h5ad(raw_file_name)
adata_sp_svc = sc.read_h5ad(sp_SVC_file_name)


### Pathway: EMT (Epithelial-Mesenchymal Transition)

1. **Download Gene Set**
   - Download the "H: hallmark gene sets" GMT file from [GSEA MSigDB](https://www.gsea-msigdb.org/gsea/msigdb/download_file.jsp?filePath=/msigdb/release/2026.1.Hs/h.all.v2026.1.Hs.symbols.gmt)

2. **Setup Directory**
   - Create a directory named `[pathway]`
   - Move the downloaded GMT file into this directory

In [ ]:
gmt_file_path = './pathway/h.all.v2025.1.Hs.symbols.gmt'
pathway_dict = read_gmt(gmt_file_path)
print(len(pathway_dict))

select_sp_svc = sample_by_identity(adata_sp_svc, sample_size=30000, seed=42)

In [ ]:
pathway_name = 'HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION'
pathway_dict = {pathway_name: pathway_dict[pathway_name]} # filter 

score_method = "AUC"
save_dir = f"../../output/sp_SVC_analysis/{patient_id}_{data_type}"

In [ ]:
select_sp_svc, pathway_score_key = score_gene_set_aucell(
    select_sp_svc, pathway_dict[pathway_name], score_name=pathway_name
)
os.makedirs(f"{save_dir}/sp_SVC", exist_ok=True)
sc.pl.scatter(select_sp_svc, x="x", y="y", color=pathway_score_key, size=40)
plt.savefig(f"{save_dir}/sp_SVC/{pathway_name}.png")
plt.show()

We found in Fibroblast, gene COMP, SFRP4 and SULF1 show high correlation with EMT pathway score.

In [ ]:
genes = ["COMP", "SFRP4", "SULF1"]

select_ct = "Fibroblast"
adata_sp = adata_sp[adata_sp.obs["Level1"] == select_ct]

adata_sp_svc = adata_sp_svc[adata_sp_svc.obs["Level1"] == select_ct].copy()
sc.pp.normalize_total(adata_sp_svc, target_sum=1e4)
sc.pp.log1p(adata_sp_svc)

In [ ]:
cmap = "Reds"
cmap = None
size = 10

In [ ]:
adata_sp_genes = adata_sp[:, genes]
adata_sp_svc_genes = adata_sp_svc[:, genes]

print("Original adata gene spatial expression:")
for gene in genes:
    sc.pl.scatter(adata_sp_genes, x="x", y="y", 
                  color=gene, 
                  size=size,
                  color_map=cmap,
                  )
print("sp_SVC adata gene spatial expression:")
for gene in genes:
    sc.pl.scatter(adata_sp_svc_genes, x="x", y="y", 
                  color=gene, 
                  size=size,
                  color_map=cmap,
                  )


## Dotplot analysis to visualize marker specificity

In [ ]:
import scanpy as sc

raw_data_path = "../../raw_data/Real_application"
svc_data_path = "../../output/sp_SVC_case"

patient_id = "P1CRC"
data_type = "HD"

raw_file_name = f"{raw_data_path}/{patient_id}_{data_type}.h5ad"
sp_SVC_file_name = f"{svc_data_path}/{patient_id}/sp_SVC.h5ad"
adata_sp = sc.read_h5ad(raw_file_name)
adata_sp_svc = sc.read_h5ad(sp_SVC_file_name)


In [ ]:
adata_sp, adata_sp_svc = sample_paired_by_identity(
    adata_sp, adata_sp_svc, sample_size=30000, seed=42
)

In [ ]:
gene_list = [
    # T cells
    "CD3D", "CD3E", "CD2", "TRAC", "TRBC1", "CD4", "CD8A", "CD8B", "IL7R",
    # B cells
    "MS4A1", "CD19", "CD79A", "CD79B", "PAX5",
    # Plasma cells
    "MZB1", "JCHAIN", "XBP1", "PRDM1", "IGKC",
    # NK cells
    "KLRD1", "NKG7", "GNLY", "PRF1", "GZMB", "GZMA",
    # Monocytes
    "CD14", "FCGR3A", "S100A8", "S100A9", "LYZ",
    # Macrophages
    "CD68", "CD163", "MARCO", "MRC1", "MSR1",
    # Dendritic cells
    "CLEC9A", "ITGAX", "CD1C", "LAMP3", "IRF8",
    # Mast cells
    "TPSAB1", "TPSB2", "KIT", "CMA1",
    # Vascular endothelial
    "PECAM1", "VWF", "CDH5", "KDR", "CLDN5",
    # Lymphatic endothelial
    "PROX1", "LYVE1", "PDPN", "FLT4",
    # Fibroblasts
    "COL1A1", "COL1A2", "COL3A1", "DCN", "LUM", "FAP", "PDGFRA",
    # Pericytes
    "RGS5", "PDGFRB", "CSPG4", "ACTA2",
    # Smooth muscle cells
    "ACTA2", "TAGLN", "MYH11", "CNN1",
    # Intestinal epithelial
    "EPCAM", "KRT19", "KRT8", "MUC2", "TFF3", "CDH1",
    # Tumor cells (general)
    "EPCAM", "KRT18", "KRT8", "MKI67", "SOX2"
]



## Marker Program Visualization

Show marker-gene dot plots for raw and sp-SVC expression layers.


In [ ]:
selected_marker_genes = {
    # 'T': [ "CD2", "TRBC1",],
    'T': [ "CD3D", "CD3E",],
    'B': [ "CD79B", "MS4A1"],
    'Plasma': ["JCHAIN", "IGKC"],
    'Lymphatic EC': ["PROX1", "FLT4"],
    'SMC': ["CNN1","MYH11"],
    'Fibroblast': [ "LUM", "COL1A1"],
    'Pericyte': ["RGS5", "NOTCH3"],
    'Mast': ["KIT"],
    'Intestinal Epithelial': ["MUC2"],
    'Monocytes': ["CD14"],
    'DC': ["LAMP3",],
}


In [ ]:
def add_cutoff_layer(adata, cutoff = 2):
    adata.layers['cutoff'] = adata.X.copy()
    adata.layers['cutoff'][adata.layers['cutoff'] > cutoff] = cutoff

    return adata

In [ ]:
cutoff = 1
adata_sp = add_cutoff_layer(adata_sp, cutoff=cutoff)
adata_sp_svc = add_cutoff_layer(adata_sp_svc, cutoff=cutoff)


In [ ]:
desired_order = ['T', 'B', 'Plasma', 'Lymphatic EC',  
                 'SMC', 'Fibroblast', 'Pericyte', 
                  'Mast', 
                  'Intestinal Epithelial','Mono_Macro','DC', 
                'Gliacyte',  'Tumor','Vascular EC', 
     ]
adata_sp.obs['Level1'].replace('Mono/Macro', 'Mono_Macro', inplace=True)
adata_sp = adata_sp[adata_sp.obs['Level1'].isin(desired_order)]
adata_sp.obs['Level1'] = adata_sp.obs['Level1'].cat.reorder_categories(desired_order)

adata_sp_svc = adata_sp_svc[adata_sp_svc.obs['Level1'].isin(desired_order)]
adata_sp_svc.obs['Level1'] = adata_sp_svc.obs['Level1'].cat.reorder_categories(desired_order)

In [ ]:
import matplotlib.pyplot as plt

groupby = "Level1"
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))


cmap = 'YlOrRd'

sc.pl.dotplot(adata_sp, selected_marker_genes, groupby, 
              layer="cutoff",
              dendrogram=False, ax=ax1, show=False,
              cmap=cmap)
ax1.set_title(f'Raw Data - {patient_id}_{data_type}')

sc.pl.dotplot(adata_sp_svc, selected_marker_genes, groupby, 
              layer="cutoff",
              dendrogram=False, ax=ax2, show=False,
              cmap=cmap)
ax2.set_title(f'SVC Processed Data - {patient_id}_{data_type}')

plt.tight_layout()
plt.show()

In [ ]:
# save
save_dir = '../../output/sp_SVC_analysis'
cmap = 'YlOrRd'


# plt.figure(figsize=(10, 6))         
sc.pl.dotplot(adata_sp, selected_marker_genes, groupby, 
              layer="cutoff",
              dendrogram=False, show = False,
              cmap = cmap,
              figsize=(7, 6)
              )
plt.savefig(f"{save_dir}/{patient_id}_{data_type}/raw_dotplot.pdf")
plt.show()
plt.close()

# plt.figure(figsize=(10, 6))         
sc.pl.dotplot(adata_sp_svc, selected_marker_genes, groupby,
              layer="cutoff",
              dendrogram=False, show = False,
              cmap = cmap,
              figsize=(7, 6)
              )
plt.savefig(f"{save_dir}/{patient_id}_{data_type}/svc_dotplot.pdf")
plt.show()
plt.close()
